# 🏥 Healthcare Operations, Revenue Optimization & Predictive Analytics
**Author:** Sambita Dutta  
**Dataset:** Cleaned Healthcare Data – 54,860 Patient Records  
**Stack:** Python · Pandas · Scikit-Learn · Plotly · Streamlit

---
### Notebook Structure
1. Environment Setup & Imports
2. Data Loading & Feature Engineering
3. Exploratory Data Analysis (EDA)
4. ML Engine 1 – Test Outcome Classifier
5. ML Engine 2 – Billing & Daily Cost Regressors
6. Patient Inference Engine (Demo)
7. Streamlit App Layout Reference


## 1. Environment Setup & Imports

In [ ]:
# ── Install dependencies (run once if needed) ─────────────────────────────
# !pip install pandas numpy scikit-learn plotly matplotlib seaborn streamlit

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ All imports successful')

## 2. Data Loading & Feature Engineering

In [ ]:
# ── Load raw CSV ───────────────────────────────────────────────────────────
df = pd.read_csv('cleaned_healthcare_data.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# ── Data types & missing values ────────────────────────────────────────────
print('--- dtypes ---')
print(df.dtypes)
print('\n--- Missing values ---')
print(df.isnull().sum())

In [ ]:
# ── Datetime feature engineering ───────────────────────────────────────────
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce')
df['Discharge Date']    = pd.to_datetime(df['Discharge Date'],    errors='coerce')

df['Admission_Year']      = df['Date of Admission'].dt.year
df['Admission_Month']     = df['Date of Admission'].dt.month
df['Admission_Quarter']   = df['Date of Admission'].dt.quarter
df['Admission_DayOfWeek'] = df['Date of Admission'].dt.dayofweek  # 0 = Monday
df['Admission_MonthName'] = df['Date of Admission'].dt.strftime('%b %Y')

# ── Anomaly flags (90th-percentile thresholds) ────────────────────────────
LOS_THRESH  = df['Length_of_Stay'].quantile(0.90)
COST_THRESH = df['Daily_Cost'].quantile(0.90)
df['LOS_Outlier']   = df['Length_of_Stay'] > LOS_THRESH
df['HighCost_Flag'] = df['Daily_Cost']     > COST_THRESH

print(f'LOS 90th-pct threshold  : {LOS_THRESH:.1f} days')
print(f'Cost 90th-pct threshold : ${COST_THRESH:,.2f}/day')
print(f'LOS outliers flagged    : {df["LOS_Outlier"].sum():,}')
print(f'High-cost rows flagged  : {df["HighCost_Flag"].sum():,}')
df.head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Numerical summary ──────────────────────────────────────────────────────
df[['Age','Length_of_Stay','Billing Amount','Daily_Cost']].describe().round(2)

In [ ]:
# ── Distribution plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cols = ['Age', 'Length_of_Stay', 'Billing Amount', 'Daily_Cost']
colors = ['#3b82f6','#10b981','#f59e0b','#ef4444']
for ax, col, color in zip(axes.flat, cols, colors):
    ax.hist(df[col].dropna(), bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Distribution of Key Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical distributions ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['Medical Condition', 'Admission Type', 'Test Results']):
    vc = df[col].value_counts()
    ax.bar(vc.index, vc.values, color='#3b82f6', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── LOS Heatmap: Medical Condition × Admission Type ────────────────────────
pivot = (
    df.groupby(['Medical Condition', 'Admission Type'])['Length_of_Stay']
    .mean()
    .reset_index()
    .pivot(index='Medical Condition', columns='Admission Type', values='Length_of_Stay')
)
plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', linewidths=0.5, cbar_kws={'label': 'Avg LOS (days)'})
plt.title('Avg Length of Stay: Medical Condition × Admission Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Billing distribution by Insurance Provider ─────────────────────────────
plt.figure(figsize=(12, 5))
order = df.groupby('Insurance Provider')['Billing Amount'].median().sort_values().index
sns.boxplot(data=df, x='Insurance Provider', y='Billing Amount', order=order,
            palette='pastel', flierprops=dict(marker='.', alpha=0.3))
plt.title('Billing Amount by Insurance Provider', fontsize=13, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Admission trends (quarterly) ───────────────────────────────────────────
df['_qtr'] = df['Admission_Year'].astype(str) + ' Q' + df['Admission_Quarter'].astype(str)
trend = df.groupby(['_qtr', 'Admission Type']).size().reset_index(name='Count').sort_values('_qtr')
fig = px.line(trend, x='_qtr', y='Count', color='Admission Type', markers=True,
              title='Quarterly Admission Trends by Admission Type',
              labels={'_qtr': 'Quarter', 'Count': '# Admissions'},
              color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=420, xaxis_tickangle=-45)
fig.show()
df.drop(columns=['_qtr'], inplace=True, errors='ignore')

In [ ]:
# ── Daily Cost by Medical Condition × Age Group ────────────────────────────
age_order = ['Child', 'Adult', 'Middle-Aged', 'Senior']
cost_pivot = (
    df.groupby(['Medical Condition', 'Age Group'])['Daily_Cost']
    .mean().reset_index()
)
cost_pivot['Age Group'] = pd.Categorical(cost_pivot['Age Group'], categories=age_order, ordered=True)
fig = px.bar(cost_pivot.sort_values('Age Group'),
             x='Medical Condition', y='Daily_Cost', color='Age Group',
             barmode='group', title='Avg Daily Cost by Condition & Age Group',
             labels={'Daily_Cost': 'Avg Daily Cost ($)'},
             color_discrete_sequence=px.colors.qualitative.Set1,
             category_orders={'Age Group': age_order})
fig.update_layout(height=400)
fig.show()

## 4. ML Engine 1 – Test Outcome Classifier

In [ ]:
# ── Feature sets ───────────────────────────────────────────────────────────
CLF_NUMERIC = ['Age', 'Length_of_Stay']
CLF_ORDINAL = ['Age Group']
CLF_NOMINAL = ['Gender', 'Medical Condition', 'Admission Type', 'Medication']
AGE_ORDER   = [['Child', 'Adult', 'Middle-Aged', 'Senior']]

feature_cols = CLF_NUMERIC + CLF_ORDINAL + CLF_NOMINAL
sub = df[feature_cols + ['Test Results']].dropna()

le = LabelEncoder()
y  = le.fit_transform(sub['Test Results'])
X  = sub[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print('Classes:', le.classes_)

In [ ]:
# ── Build & train classifier pipeline ─────────────────────────────────────
clf_preprocessor = ColumnTransformer([
    ('num', 'passthrough',                              CLF_NUMERIC),
    ('ord', OrdinalEncoder(categories=AGE_ORDER),       CLF_ORDINAL),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CLF_NOMINAL),
], remainder='drop')

clf_pipeline = Pipeline([
    ('prep',  clf_preprocessor),
    ('model', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.08, max_depth=4,
        random_state=42, subsample=0.8
    )),
])

print('Training classifier ...')
clf_pipeline.fit(X_train, y_train)
print('Done.')

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
y_pred_clf = clf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_clf, target_names=le.classes_, zero_division=0))

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_clf)
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix – Test Outcome Classifier', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance ─────────────────────────────────────────────────────
ohe_clf     = clf_pipeline.named_steps['prep'].named_transformers_['nom']
nom_names   = ohe_clf.get_feature_names_out(CLF_NOMINAL).tolist()
feat_names  = CLF_NUMERIC + CLF_ORDINAL + nom_names
importances = clf_pipeline.named_steps['model'].feature_importances_

fi_clf = (
    pd.DataFrame({'Feature': feat_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
    .head(15)
)

plt.figure(figsize=(9, 5))
plt.barh(fi_clf['Feature'][::-1], fi_clf['Importance'][::-1], color='#3b82f6')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances – Classifier', fontweight='bold')
plt.tight_layout()
plt.show()
fi_clf

## 5. ML Engine 2 – Billing & Daily Cost Regressors

In [ ]:
# ── Feature sets ───────────────────────────────────────────────────────────
REG_NUMERIC = ['Age', 'Length_of_Stay']
REG_NOMINAL = ['Gender', 'Medical Condition', 'Admission Type',
               'Medication', 'Insurance Provider', 'Age Group']

reg_preprocessor = ColumnTransformer([
    ('num', 'passthrough',                                              REG_NUMERIC),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), REG_NOMINAL),
], remainder='drop')

reg_feature_cols = REG_NUMERIC + REG_NOMINAL
sub_reg = df[reg_feature_cols + ['Billing Amount', 'Daily_Cost']].dropna()
X_reg = sub_reg[reg_feature_cols]
print(f'Regression dataset: {X_reg.shape}')

In [ ]:
# ── Train & evaluate both regressors ──────────────────────────────────────
reg_results = {}

for target in ['Billing Amount', 'Daily_Cost']:
    y_reg = sub_reg[target].values
    Xtr, Xte, ytr, yte = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

    from sklearn.compose import ColumnTransformer as CT
    reg_prep = CT([
        ('num', 'passthrough', REG_NUMERIC),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), REG_NOMINAL),
    ], remainder='drop')

    pipe = Pipeline([
        ('prep',  reg_prep),
        ('model', GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.08, max_depth=4,
            random_state=42, subsample=0.8
        )),
    ])
    pipe.fit(Xtr, ytr)
    ypred = pipe.predict(Xte)

    rmse = np.sqrt(mean_squared_error(yte, ypred))
    mae  = mean_absolute_error(yte, ypred)
    r2   = r2_score(yte, ypred)

    ohe      = pipe.named_steps['prep'].named_transformers_['nom']
    fn_names = REG_NUMERIC + ohe.get_feature_names_out(REG_NOMINAL).tolist()
    fi_df    = (
        pd.DataFrame({'Feature': fn_names, 'Importance': pipe.named_steps['model'].feature_importances_})
        .sort_values('Importance', ascending=False).head(15)
    )
    reg_results[target] = {'pipeline': pipe, 'rmse': rmse, 'mae': mae, 'r2': r2, 'fi': fi_df, 'ypred': ypred, 'yte': yte}
    print(f'[{target}]  RMSE=${rmse:,.0f}  MAE=${mae:,.0f}  R²={r2:.3f}')

In [ ]:
# ── Feature importance plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {'Billing Amount': '#f59e0b', 'Daily_Cost': '#ef4444'}
for ax, target in zip(axes, ['Billing Amount', 'Daily_Cost']):
    fi = reg_results[target]['fi']
    ax.barh(fi['Feature'][::-1], fi['Importance'][::-1], color=colors[target])
    ax.set_title(f'Top Features – {target}', fontweight='bold')
    ax.set_xlabel('Importance')
plt.suptitle('Regressor Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Actual vs Predicted scatter ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, target in zip(axes, ['Billing Amount', 'Daily_Cost']):
    yte   = reg_results[target]['yte']
    ypred = reg_results[target]['ypred']
    ax.scatter(yte, ypred, alpha=0.25, s=8, color='#3b82f6')
    lim = [min(yte.min(), ypred.min()), max(yte.max(), ypred.max())]
    ax.plot(lim, lim, 'r--', lw=1.5)
    ax.set_xlabel(f'Actual {target}')
    ax.set_ylabel(f'Predicted {target}')
    ax.set_title(f'{target} – Actual vs Predicted (R²={reg_results[target]["r2"]:.3f})',
                 fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Patient Inference Engine (Demo)

In [ ]:
# ── Helper: age → age group ────────────────────────────────────────────────
def age_to_group(age):
    if age < 18:   return 'Child'
    elif age < 45: return 'Adult'
    elif age < 65: return 'Middle-Aged'
    else:          return 'Senior'

def predict_patient_nb(
    age, gender, admission_type, condition, insurance, medication, planned_days
):
    age_group  = age_to_group(age)
    bill_pipe  = reg_results['Billing Amount']['pipeline']
    cost_pipe  = reg_results['Daily_Cost']['pipeline']

    # ── Classification ───────────────────────────────────────────────────
    clf_in = pd.DataFrame([{
        'Age': age, 'Length_of_Stay': planned_days, 'Age Group': age_group,
        'Gender': gender, 'Medical Condition': condition,
        'Admission Type': admission_type, 'Medication': medication,
    }])
    proba  = clf_pipeline.predict_proba(clf_in)[0]
    proba_dict = {cls: round(float(p), 3) for cls, p in zip(le.classes_, proba)}

    # ── Regression ──────────────────────────────────────────────────────
    reg_in = pd.DataFrame([{
        'Age': age, 'Length_of_Stay': planned_days, 'Gender': gender,
        'Medical Condition': condition, 'Admission Type': admission_type,
        'Medication': medication, 'Insurance Provider': insurance,
        'Age Group': age_group,
    }])
    pred_billing = float(bill_pipe.predict(reg_in)[0])
    pred_cost    = float(cost_pipe.predict(reg_in)[0])

    # ── Anomaly flags ────────────────────────────────────────────────────
    los_thresh  = df['Length_of_Stay'].quantile(0.90)
    cost_thresh = df['Daily_Cost'].quantile(0.90)
    cond_df     = df[df['Medical Condition'] == condition]
    cond_los    = cond_df['Length_of_Stay'].mean() if not cond_df.empty else df['Length_of_Stay'].mean()
    cond_cost   = cond_df['Daily_Cost'].mean()     if not cond_df.empty else df['Daily_Cost'].mean()

    return {
        'proba_dict':      proba_dict,
        'predicted_class': le.classes_[proba.argmax()],
        'pred_billing':    round(pred_billing, 2),
        'pred_daily_cost': round(pred_cost, 2),
        'los_risk':        planned_days > los_thresh,
        'cost_risk':       pred_cost   > cost_thresh,
        'above_cond_los':  planned_days > cond_los  * 1.5,
        'above_cond_cost': pred_cost    > cond_cost * 1.5,
        'cond_los_mean':   round(cond_los, 1),
        'cond_cost_mean':  round(cond_cost, 2),
    }

# ── Example inference ─────────────────────────────────────────────────────
result = predict_patient_nb(
    age=55, gender='Female', admission_type='Emergency',
    condition='Cancer', insurance='Medicare',
    medication='Aspirin', planned_days=10
)
for k, v in result.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# ── Visualise inference probabilities ────────────────────────────────────
proba_vals = list(result['proba_dict'].values())
proba_labs = list(result['proba_dict'].keys())
colors_map = {'Normal': '#22c55e', 'Abnormal': '#ef4444', 'Inconclusive': '#f59e0b'}

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(proba_labs, proba_vals, color=[colors_map.get(l,'#3b82f6') for l in proba_labs], edgecolor='white')
for bar, val in zip(bars, proba_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_ylabel('Probability')
ax.set_title(f'Predicted Test Outcome: {result["predicted_class"]}', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Streamlit App Layout Reference

The interactive dashboard is implemented in `app.py` (backed by `data_pipeline.py`).

```
streamlit run app.py
```

### Tab Structure

| Tab | Contents |
|-----|----------|
| 📊 Executive & Operational | KPIs · LOS Heatmap · Hospital Workload · Admission Trends |
| 💰 Financial & Revenue Cycle | KPIs · Billing Violin · Anomaly Table · Daily Cost by Age |
| 🧬 Clinical Outcomes & ML | Test Result Charts · Model Metrics · Patient Inference Form |

### Sidebar Filters
- Admission Date Range  
- Insurance Provider (multi-select)  
- Hospital (multi-select)  
- Admission Type (multi-select)  
- Medical Condition (multi-select)

All visuals and KPIs update live as filters are applied.
